# GOBU Buddy — Knowledge Chunk Builder

Notebook ini menyiapkan knowledge base GOBU menjadi beberapa chunk agar mudah dicari oleh sistem RAG. Notebook tidak memerlukan OpenAI API.

**Cara pakai:** upload file `knowledge_source.json` dari folder `ai/` project GOBU, jalankan semua cell, lalu unduh `knowledge_chunks.json` dan letakkan kembali ke folder `data/` project.

Model inference untuk demo lokal dijalankan dengan Ollama (`qwen2.5:3b`), sedangkan notebook ini dipakai untuk mengelola, memecah, dan memvalidasi materi. Jangan memasukkan informasi supplier/proses rahasia ke dokumen publik.

In [ ]:
from google.colab import files
import json, re, hashlib
from pathlib import Path

uploaded = files.upload()
source_name = next((name for name in uploaded if name.endswith('.json')), None)
if not source_name:
    raise ValueError('Upload ai/knowledge_source.json terlebih dahulu.')
source_docs = json.loads(Path(source_name).read_text(encoding='utf-8'))
print(f'{len(source_docs)} dokumen sumber dimuat.')

In [ ]:
# Pengaturan chunking
CHUNK_SIZE = 900       # karakter maksimum per chunk
OVERLAP = 120          # overlap untuk menjaga konteks antar-chunk

def split_text(text, chunk_size=CHUNK_SIZE, overlap=OVERLAP):
    text = re.sub(r'\s+', ' ', text).strip()
    if len(text) <= chunk_size:
        return [text]
    parts, start = [], 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        if end < len(text):
            boundary = text.rfind('. ', start, end)
            if boundary > start + int(chunk_size * 0.55):
                end = boundary + 1
        piece = text[start:end].strip()
        if piece:
            parts.append(piece)
        if end >= len(text):
            break
        start = max(end - overlap, start + 1)
    return parts

chunks = []
for doc in source_docs:
    pieces = split_text(doc['content'])
    for index, piece in enumerate(pieces, start=1):
        chunk_id = f"{doc['id']}-{index:02d}"
        chunks.append({
            'id': chunk_id,
            'parent_id': doc['id'],
            'category': doc['category'],
            'title': doc['title'],
            'keywords': doc.get('keywords', []),
            'content': piece,
            'source': 'GOBU owner-provided knowledge',
            'version': '1.0'
        })

print(f'{len(source_docs)} dokumen menjadi {len(chunks)} chunks.')

In [ ]:
# Validasi: ID unik, kolom wajib tersedia, dan tidak ada konten kosong
required = {'id', 'parent_id', 'category', 'title', 'content', 'source', 'version'}
ids = [c['id'] for c in chunks]
assert len(ids) == len(set(ids)), 'Ada ID chunk duplikat.'
assert all(required.issubset(c.keys()) for c in chunks), 'Ada field wajib yang hilang.'
assert all(c['content'].strip() for c in chunks), 'Ada chunk kosong.'

from collections import Counter
print('Validasi berhasil.')
print('Jumlah per kategori:')
for category, count in Counter(c['category'] for c in chunks).items():
    print(f'- {category}: {count}')

In [ ]:
# Ekspor file yang akan digunakan oleh GOBU Buddy lokal
output = {
    'version': '1.0',
    'chunk_count': len(chunks),
    'chunking': {'chunk_size_chars': CHUNK_SIZE, 'overlap_chars': OVERLAP},
    'chunks': chunks
}
output_path = Path('knowledge_chunks.json')
output_path.write_text(json.dumps(output, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'File siap: {output_path} ({output_path.stat().st_size:,} bytes)')
files.download(str(output_path))

## Opsional: uji model open-source di Colab

Cell berikut menguji kemampuan dasar model Qwen2.5 0.5B Instruct dengan satu chunk knowledge. Ini hanya smoke test; aplikasi website lokal menggunakan `qwen2.5:3b` melalui Ollama agar tidak bergantung pada sesi Colab yang dapat terputus.

In [ ]:
# Jalankan bila ingin menguji model di Colab (GPU runtime direkomendasikan).
# !pip -q install transformers accelerate
# from transformers import AutoTokenizer, AutoModelForCausalLM
# import torch
# model_id = 'Qwen/Qwen2.5-0.5B-Instruct'
# tokenizer = AutoTokenizer.from_pretrained(model_id)
# model = AutoModelForCausalLM.from_pretrained(
#     model_id, torch_dtype='auto', device_map='auto'
# )
# sample = chunks[0]
# messages = [
#   {'role':'system','content':'Jawab hanya berdasarkan knowledge yang diberikan. Jika tidak tahu, katakan belum tersedia.'},
#   {'role':'user','content':f"Knowledge: {sample['content']}\n\nPertanyaan: Apa identitas brand GOBU Coffee?"}
# ]
# prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
# inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
# output = model.generate(**inputs, max_new_tokens=160, do_sample=False)
# print(tokenizer.decode(output[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True))